In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from nn_from_scratch import Sequential, Dense, ReLU, CategoricalCrossEntropy, Adam

def main():
    print("Generating synthetic 10-class classification dataset...")
    X, y = make_classification(
        n_samples=5000, 
        n_features=64, 
        n_informative=40, 
        n_classes=10, 
        random_state=42
    )
    
    # Scale features and convert labels to one-hot encoding
    X = (X - np.mean(X, axis=0)) / (np.std(X, axis=0) + 1e-8)
    encoder = OneHotEncoder(sparse_output=False)
    y_one_hot = encoder.fit_transform(y.reshape(-1, 1))
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y_one_hot, test_size=0.2, random_state=42
    )
    
    print(f"Train samples: {X_train.shape[0]}, Validation samples: {X_val.shape[0]}")
    print("Building model architecture...\n")
    
    # Define MLP Architecture
    model = Sequential([
        Dense(in_features=64, out_features=128, init_method='he'),
        ReLU(),
        Dense(in_features=128, out_features=64, init_method='he'),
        ReLU(),
        Dense(in_features=64, out_features=10, init_method='xavier')
    ])
    
    loss_fn = CategoricalCrossEntropy()
    optimizer = Adam(lr=0.005)
    
    print("Starting training loop...")
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=64,
        loss_fn=loss_fn,
        optimizer=optimizer,
        val_data=(X_val, y_val)
    )
    
    # Final evaluation
    val_preds = model.predict(X_val)
    final_acc = np.mean(np.argmax(val_preds, axis=1) == np.argmax(y_val, axis=1)) * 100
    print(f"\nFinal Validation Accuracy: {final_acc:.2f}%")

if __name__ == '__main__':
    main()

Generating synthetic 10-class classification dataset...
Train samples: 4000, Validation samples: 1000
Building model architecture...

Starting training loop...
Epoch   1/50 - Train Loss: 2.0638 | Val Loss: 1.7829 | Val Acc: 37.00%
Epoch  10/50 - Train Loss: 0.0764 | Val Loss: 1.7643 | Val Acc: 60.10%
Epoch  20/50 - Train Loss: 0.0030 | Val Loss: 2.0986 | Val Acc: 61.40%
Epoch  30/50 - Train Loss: 0.0013 | Val Loss: 2.2798 | Val Acc: 61.70%
Epoch  40/50 - Train Loss: 0.0006 | Val Loss: 2.4177 | Val Acc: 62.10%
Epoch  50/50 - Train Loss: 0.0004 | Val Loss: 2.5384 | Val Acc: 61.90%

Final Validation Accuracy: 61.90%
